In [1]:
from pathlib import Path
import pandas as pd
def load_csv_files():
    path_set = Path.cwd()
    file_names = ['header', 'line', 'premium']
    dataframes = []
    for dataset in file_names:
        files = sorted(list(path_set.glob(f'*{dataset}*.csv')))
        if files:
            db_file = files[-1]
            if db_file.stat().st_size == 0:   #This check the file is empty or not.
                raise ValueError(f"ERROR: {db_file.name} is empty")

            df = pd.read_csv(db_file)         #Reads the dataset.

            if df.empty:                       #This check the file has data in it or not even if column present.
                raise ValueError(f"ERROR: {db_file.name} has no data")
            print(f"\nLoaded {dataset} file: {db_file.name}")
            display(df.head())
            dataframes.append(df)
        else:
            raise FileNotFoundError(f"ERROR: No {dataset} CSV file found")
    return dataframes

def inspect_dataframes(df_header,df_line,df_premium):
    dataframes = [df_header.copy(),df_line.copy(),df_premium.copy()]
    names = ['HEADER', 'LINE', 'PREMIUM']
    print("\nINSPECTING DATASETS")
    for i in range(len(dataframes)):
        name = names[i]
        df = dataframes[i]
        print("*"*30)
        print(f"INSPECTING {name} DATASET")
        print("*"*30)

        # Dataset shape
        print("\nShape:")
        display(df.shape)

        # Statistical description
        print("\nDescription:")
        display(df.describe(include='all'))

        # Data types
        print("\nData Types:")
        display(df.dtypes)

        # Unique Values
        print("\nUnique Values:")
        display(df.nunique())

        # Duplicate records
        print("\nDuplicate Records:")
        display(df.duplicated().sum())

        # Missing values
        print("\nMissing Values:")
        display(df.isna().sum())

    return dataframes
def clean_dataframes(df1,df2,df3):
  #For header.
  #--> claim_id, member_id , provider_id, claim_type, place_of_service, all dates, pd1, sd2, total_billed, total_allowed, total_paid, member_response, claim_status
    # HEADER DATASET
    df1.drop(columns=['submission_channel', 'original_claim_id'],inplace=True)

    df1.dropna(subset=['claim_id','member_id','service_from_date','service_to_date','received_date','processed_date'],inplace=True)

    df1 = df1[df1['claim_status'] != 'Adjusted'].copy()

    df1 = df1.fillna("NA")

    df1.drop_duplicates(subset=['claim_id','member_id','provider_id'],inplace=True)

    print("HEADER DATASET AFTER CLEANING")
    display(df1.isna().sum())
    display(df1.nunique())
    display(df1)
  #--.> claim_id,procedure_code,billed_amount,allowed_amount,paid_amount,line_status.
  #LINE DATASE
    df2.drop(columns=['line_number','modifier','units'],inplace=True)

    df2.dropna(subset=['claim_id','procedure_code','billed_amount','allowed_amount','paid_amount'],inplace=True)

    df2 = df2[df2['line_status'] != 'Pending'].copy()

    df2.drop_duplicates(subset=['claim_id','procedure_code'],inplace=True)

    print("LINE DATASET AFTER CLEANING")
    display(df2.isna().sum())
    display(df2.nunique())
    display(df2)
  #--> member_id,premium_month,plan_id,premium_amount,payment_status.
  #PREMIUM.
    df3.dropna(subset=['member_id','premium_month','plan_id','premium_amount'],inplace=True)

    df3.drop_duplicates(subset=['member_id'],inplace=True)

    print("PREMIUM DATASET AFTER CLEANING")
    display(df3.isna().sum())
    display(df3.nunique())
    display(df3)
    return df1, df2, df3

header, line, premium = load_csv_files()
df1, df2, df3 = inspect_dataframes(header,line,premium)
# Function call
df1, df2, df3 = clean_dataframes(df1, df2, df3)
df1.to_csv('cleaned_header.csv', index=False)
df2.to_csv('cleaned_line.csv', index=False)
df3.to_csv('cleaned_premium.csv', index=False)
#


Loaded header file: claims_header_2026_01.csv


,claim_id,member_id,provider_id,claim_type,place_of_service,service_from_date,service_to_date,received_date,processed_date,primary_diagnosis,secondary_diagnosis,total_billed,total_allowed,total_paid,member_responsibility,claim_status,denial_code,submission_channel,original_claim_id
0,CLM1070278,M105444,P2070,Professional,11 - Office,2025-04-08,2025-04-08,2025-10-19,2026-01-24,S83.511A,NaN,734.18,0.00,0.00,0.00,Denied,TF01,Paper,NaN
1,CLM1009377,M100747,P2058,Professional,11 - Office,2025-08-14,2025-08-14,2025-10-20,2026-01-01,R10.9,NaN,477.47,273.86,191.71,82.15,Paid,NaN,Paper,NaN
2,CLM1001951,M100172,P2150,Professional,11 - Office,2025-10-27,2025-10-27,2025-10-31,2026-01-07,G43.909,NaN,573.78,392.08,352.88,39.20,Paid,NaN,Paper,NaN
3,CLM1087922,M106832,P2043,Institutional,23 - Emergency Room,2025-10-24,2025-10-24,2025-11-03,2026-01-25,R10.9,NaN,3860.38,2007.32,1605.87,401.46,Paid,NaN,Paper,NaN
4,CLM1097126,M107552,P2027,Professional,11 - Office,2025-10-02,2025-10-02,2025-11-03,2026-01-01,D48.5,NaN,1102.44,449.81,382.34,67.47,Paid,NaN,Paper,NaN



Loaded line file: claim_lines_2026_01.csv


,claim_id,line_number,procedure_code,modifier,units,billed_amount,allowed_amount,paid_amount,line_status
0,CLM1000013,1,99213,NaN,1,514.87,281.12,253.00,Paid
1,CLM1000013,2,80061,NaN,1,72.12,49.52,44.57,Paid
2,CLM1000028,1,97110,GP,4,603.53,338.10,304.28,Paid
3,CLM1000028,2,97140,GP,1,141.81,76.41,68.77,Paid
4,CLM1000043,1,90834,NaN,1,596.75,325.34,292.80,Paid



Loaded premium file: premiums_2026_01.csv


,member_id,premium_month,plan_id,premium_amount,payment_status
0,M100001,2026-01,PLN03,420,Received
1,M100002,2026-01,PLN08,370,Received
2,M100003,2026-01,PLN06,720,Received
3,M100004,2026-01,PLN07,960,Received
4,M100006,2026-01,PLN02,400,Received


... (44 outputs left out)

PREMIUM DATASET AFTER CLEANING


,0
member_id,0
premium_month,0
plan_id,0
premium_amount,0
payment_status,0


,0
member_id,6928
premium_month,1
plan_id,8
premium_amount,8
payment_status,2


,member_id,premium_month,plan_id,premium_amount,payment_status
0,M100001,2026-01,PLN03,420,Received
1,M100002,2026-01,PLN08,370,Received
2,M100003,2026-01,PLN06,720,Received
3,M100004,2026-01,PLN07,960,Received
4,M100006,2026-01,PLN02,400,Received
...,...,...,...,...,...
6923,M107995,2026-01,PLN05,560,Received
6924,M107996,2026-01,PLN01,380,Received
6925,M107997,2026-01,PLN03,420,Received
6926,M107998,2026-01,PLN05,560,Received
